# Data Visualization: Rankings & Distributions
## Horizontal Bar Charts + Swarm Plots

Welcome to Class 2! Today we explore two powerful visualization techniques:
- **Horizontal Bar Charts** — ideal for rankings, long labels, and top-N comparisons
- **Swarm Plots** — show every individual data point while revealing distribution shape

## Learning Objectives
By the end of this session you will be able to:
- Build and sort horizontal bar charts for clear ranking communication
- Filter and display top-N categories without clutter
- Create swarm plots that expose individual data points
- Decide when a swarm plot beats a box or violin plot
- Interpret crowding and spread in swarm visualizations

## Class Structure (90 minutes)
| Time | Activity |
|------|----------|
| 10 min | Recap + mini-check |
| 25 min | Concept walkthrough — examples & common mistakes |
| 40 min | Guided lab — students build |
| 10 min | Challenge / extension stretch task |
| 5 min  | Wrap-up + exit ticket |

Let's begin by setting up our environment.

## Quick Recap — Class 1 Check-in

Before we dive in, let's briefly recall what we covered last class:

1. **Area Charts** — show composition over time (stacked layers)
2. **Heatmaps** — reveal correlation patterns using color intensity

### Mini-Check Questions *(think before scrolling)*
- When would you choose a stacked area chart over a grouped bar chart?
- What does a correlation value of −0.85 tell you in a heatmap?
- Why does category ordering matter in stacked area charts?

*Discuss with your neighbor for 2 minutes, then we will compare answers.*

In [ ]:
# ─── Environment Setup ───────────────────────────────────────────────────────
# We import the same core libraries used throughout this course.
# pandas   → tabular data manipulation (think of it as Excel in Python)
# numpy    → numerical operations and array handling
# matplotlib.pyplot → low-level plotting engine (foundation of all charts)
# seaborn  → high-level statistical visualization built on top of matplotlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set a consistent visual style for all plots in this notebook.
# "whitegrid" gives clean white backgrounds with subtle grid lines for readability.
sns.set_style("whitegrid")

# Set default figure dimensions (width=11 inches, height=6 inches).
# Wider figures give horizontal bars enough room to display long labels.
plt.rcParams["figure.figsize"] = (11, 6)

# Tell Jupyter to display plots directly inside the notebook (not in a new window).
%matplotlib inline

print("Environment ready! Libraries imported successfully.")
print("We are set to explore Horizontal Bar Charts and Swarm Plots.")


---
# Part 1: Horizontal Bar Charts — Communicating Rankings Clearly

A **horizontal bar chart** places categories on the Y-axis and values on the X-axis.
This simple flip from the familiar vertical bar chart unlocks several advantages:

| Challenge | Why Horizontal Bars Help |
|-----------|--------------------------|
| Long category names | Labels sit on the left — no rotation needed |
| Ranking / ordering | Sorting top-to-bottom makes hierarchy instantly visible |
| Many categories | Vertical space is easier to scroll than horizontal |
| Comparing magnitudes | Bars extend left-to-right like a natural reading direction |

## Concepts We Will Cover
1. Basic horizontal bar chart with `barh()`
2. Sorting data for clear ranking
3. Top-N filtering — showing only the most important items
4. Adding value labels directly on bars
5. Common mistakes and how to fix them

In [ ]:
# ─── Dataset: Global Programming Language Popularity (2024 Survey) ──────────
# We simulate a developer survey result: percentage of developers using each language.
# This is a realistic scenario where horizontal bars shine because of long label names.

# Dictionary mapping language name → percentage of survey respondents who use it
language_popularity = {
    "Python":                 68.4,
    "JavaScript":             65.8,
    "TypeScript":             38.5,
    "SQL":                    54.7,
    "Java":                   35.2,
    "C#":                     27.6,
    "C++":                    23.1,
    "C":                      19.8,
    "Go":                     14.3,
    "Rust":                   13.1,
    "Kotlin":                 9.4,
    "Swift":                  8.7,
    "R":                      7.9,
    "MATLAB":                 5.6,
    "Scala":                  4.2,
    "Perl":                   3.8,
    "Ruby":                   3.5,
    "PHP":                    3.1,
    "Dart":                   2.9,
    "Haskell":                2.1,
}

# Convert to a pandas DataFrame — each row is one language with its usage percentage
# pd.DataFrame.from_dict() turns a dictionary into a table
df_lang = pd.DataFrame(
    list(language_popularity.items()),     # list of (language, value) tuples
    columns=["Language", "Usage_Percent"]  # assign column names explicitly
)

# Preview the unsorted data to see what we are working with
print("Raw dataset (unsorted):")
print(df_lang.to_string(index=False))
print(f"\nTotal languages tracked: {len(df_lang)}")
print(f"Highest usage: {df_lang['Usage_Percent'].max()}% — {df_lang.loc[df_lang['Usage_Percent'].idxmax(), 'Language']}")
print(f"Lowest  usage: {df_lang['Usage_Percent'].min()}% — {df_lang.loc[df_lang['Usage_Percent'].idxmin(), 'Language']}")


In [ ]:
# ─── Basic Horizontal Bar Chart ─────────────────────────────────────────────
# This is the starting point — a plain chart BEFORE we apply best practices.
# We will improve it step by step so you can see exactly WHY each change matters.

# Step 1: Sort the DataFrame by usage in ASCENDING order.
# Ascending order puts the lowest bar at the top in a barh() chart,
# which means the HIGHEST value appears at the BOTTOM — the natural reading end.
# We will flip this in the next cell; for now let's see what unsorted looks like.
df_unsorted = df_lang.copy()   # work on a copy so we don't alter the original

plt.figure(figsize=(11, 8))

# plt.barh() draws horizontal bars.
# First argument: Y-axis labels (category names)
# Second argument: X-axis values (bar lengths)
plt.barh(df_unsorted["Language"], df_unsorted["Usage_Percent"])

# Basic labels and title
plt.title("Programming Language Popularity (Unsorted — Hard to Read)", fontsize=15)
plt.xlabel("% of Developers Using the Language", fontsize=12)
plt.ylabel("Language", fontsize=12)

plt.tight_layout()
plt.show()

print("Observation: Without sorting, it is very hard to rank languages at a glance.")
print("The eye has to jump around to compare — this is a common beginner mistake.")


In [ ]:
# ─── Sorted Horizontal Bar Chart — Best Practice ────────────────────────────
# Sorting turns a confusing chart into an instant ranking.
# Rule of thumb: sort DESCENDING by value so the winner is at the TOP.

# Sort the full dataset in descending order of Usage_Percent
# ascending=False → largest value first (top of chart)
df_sorted = df_lang.sort_values("Usage_Percent", ascending=True)
# NOTE: we use ascending=True because barh() draws from bottom to top,
# so the first row ends up at the BOTTOM of the chart.
# By sorting ascending, the largest value is last → appears at the TOP.

plt.figure(figsize=(11, 9))

# Draw the sorted horizontal bars with a pleasant steel-blue color
# color= accepts any named color, hex code, or RGB tuple
bars = plt.barh(
    df_sorted["Language"],        # Y-axis: language names in sorted order
    df_sorted["Usage_Percent"],   # X-axis: percentage values
    color="steelblue",            # bar fill color
    edgecolor="white",            # thin white border between bars for clarity
    height=0.7                    # bar height (0-1 scale; 0.7 gives small gaps)
)

# Add VALUE LABELS directly on each bar — readers should not need to squint at the axis
# enumerate() gives us both the index (i) and the bar object
for bar in bars:
    width = bar.get_width()   # get_width() returns the bar's X-axis extent = the value
    # plt.text() places a text annotation
    # x: slightly to the right of the bar end (value + 0.5 offset)
    # y: vertically centered on the bar (bar center = bar.get_y() + half bar height)
    plt.text(
        width + 0.5,                                  # x position (just right of bar tip)
        bar.get_y() + bar.get_height() / 2,           # y position (bar vertical center)
        f"{width:.1f}%",                              # label text formatted to 1 decimal
        va="center",                                  # vertical alignment: center
        ha="left",                                    # horizontal alignment: left-aligned
        fontsize=9,                                   # small font so it doesn't crowd
        color="black"
    )

# Chart decoration
plt.title("Programming Language Popularity 2024 (Sorted Ranking)", fontsize=16, pad=18)
plt.xlabel("% of Developers Using the Language", fontsize=12)
plt.ylabel("Language", fontsize=12)
plt.xlim(0, 80)          # give extra room on the right for value labels
plt.grid(axis="x", alpha=0.3)   # only show vertical grid lines (along value axis)
plt.tight_layout()
plt.show()

print("Improvement: Sorting instantly shows the ranking hierarchy.")
print("Python leads at 68.4%, followed by JavaScript (65.8%) and SQL (54.7%).")


In [ ]:
# ─── Top-N Filtering — Show Only What Matters ───────────────────────────────
# With 20 languages, the chart is getting long.
# In most presentations you only need the TOP-N items to tell your story clearly.
# Showing everything can overwhelm the audience and dilute the key message.

# Define how many top languages to display
TOP_N = 10

# Select the top-N by sorting descending and taking the first N rows
# We then re-sort ascending so the chart displays highest at top (barh convention)
df_top = (
    df_lang
    .sort_values("Usage_Percent", ascending=False)   # rank from highest to lowest
    .head(TOP_N)                                      # keep only top N rows
    .sort_values("Usage_Percent", ascending=True)    # flip for correct barh direction
)

# Create a color gradient: darker bars for higher-ranked languages
# np.linspace generates N evenly-spaced values between 0.35 and 0.85
# These are used as input to a matplotlib colormap (Blues)
color_values = np.linspace(0.35, 0.85, TOP_N)             # array of shade intensities
colors = plt.cm.Blues(color_values)                         # map to blue color palette

plt.figure(figsize=(11, 7))

bars = plt.barh(
    df_top["Language"],
    df_top["Usage_Percent"],
    color=colors,         # gradient colors: darker = higher rank
    edgecolor="white",
    height=0.65
)

# Value labels on each bar
for bar in bars:
    width = bar.get_width()
    plt.text(
        width + 0.4,
        bar.get_y() + bar.get_height() / 2,
        f"{width:.1f}%",
        va="center", ha="left", fontsize=10, fontweight="bold"
    )

# Vertical annotation line at the median of the full dataset
median_val = df_lang["Usage_Percent"].median()
plt.axvline(x=median_val, color="crimson", linestyle="--", linewidth=1.5,
            label=f"All-language median: {median_val:.1f}%")

plt.title(f"Top {TOP_N} Programming Languages by Developer Usage (2024)",
          fontsize=16, pad=18)
plt.xlabel("% of Developers", fontsize=12)
plt.ylabel("Language", fontsize=12)
plt.xlim(0, 82)
plt.legend(fontsize=10)
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Top {TOP_N} languages account for the vast majority of developer mindshare.")
print(f"The dashed line shows the all-language median ({median_val:.1f}%),")
print("revealing that most languages fall well below the most popular handful.")


In [ ]:
# ─── Common Mistakes with Horizontal Bar Charts ─────────────────────────────
# Seeing mistakes side-by-side with corrections cements best practices in memory.

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# ── MISTAKE: Unsorted + no labels + rainbow colors ────────────────────────────
# Rainbow (jet) colormap encodes no meaning — it just adds visual noise
colors_bad = plt.cm.jet(np.linspace(0, 1, len(df_lang)))
axes[0].barh(df_lang["Language"], df_lang["Usage_Percent"], color=colors_bad)
axes[0].set_title("❌ Common Mistakes:
Unsorted · No Labels · Rainbow Colors",
                  fontsize=13, color="crimson", pad=12)
axes[0].set_xlabel("Usage %")
# No value labels — reader must read the x-axis for every bar (tedious)

# ── CORRECT: Sorted + labeled + single purposeful color ──────────────────────
df_correct = df_lang.sort_values("Usage_Percent", ascending=True)
bars_good = axes[1].barh(df_correct["Language"], df_correct["Usage_Percent"],
                          color="teal", edgecolor="white", height=0.7)
for bar in bars_good:
    w = bar.get_width()
    axes[1].text(w + 0.3, bar.get_y() + bar.get_height() / 2,
                 f"{w:.1f}%", va="center", ha="left", fontsize=8)
axes[1].set_title("✅ Best Practices:
Sorted · Labeled · Single Color",
                  fontsize=13, color="green", pad=12)
axes[1].set_xlabel("Usage %")
axes[1].set_xlim(0, 82)
axes[1].grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

print("Key Rules for Horizontal Bar Charts:")
print("  1. Always sort — ascending sort gives descending visual rank in barh()")
print("  2. Label bars directly — remove the need to look at axes")
print("  3. Use a single color or a purposeful gradient (not rainbow)")
print("  4. Limit to Top-N when the full list is too long")
print("  5. Add a grid ONLY on the value axis (axis='x')")


---
# Part 2: Swarm Plots — Showing Every Individual Data Point

A **swarm plot** (also called a beeswarm plot) displays every observation as a dot,
arranging points so they don't overlap. This preserves full information while
still conveying distribution shape — something box and violin plots cannot do.

## Why Swarm Beats Box/Violin in Certain Situations

| Scenario | Box Plot | Violin Plot | Swarm Plot |
|----------|----------|-------------|------------|
| Small dataset (n < 100) | ⚠️ Hides individuals | ⚠️ Misleading smoothing | ✅ Shows all points |
| Detecting clusters / gaps | ❌ Cannot see | ⚠️ Sometimes | ✅ Clear |
| Showing outlier context | ⚠️ Only flags | ⚠️ Limited | ✅ Exact position |
| Large dataset (n > 1000) | ✅ Fast | ✅ Clean | ⚠️ Can get dense |

## Key Concepts We Will Cover
1. Basic swarm plot with `seaborn.swarmplot()`
2. Swarm + box overlay — best of both worlds
3. Crowding and `dodge` — handling multiple groups
4. Color encoding for a third variable
5. When swarm plot is not the right choice

In [ ]:
# ─── Dataset: Student Exam Scores Across Departments ────────────────────────
# We create a synthetic dataset representing final exam scores for 120 students
# across 4 university departments. This is a perfect swarm plot use-case because:
#   • Small-ish sample (120 students) — every point is meaningful
#   • We want to see individuals, not just averages
#   • We suspect department differences AND within-department variation

np.random.seed(42)   # fix random seed for reproducibility — same data every run

# Define the four departments and how many students are in each
departments = ["Computer Science", "Mathematics", "Physics", "Biology"]
n_per_dept  = [35, 28, 32, 25]   # slightly unequal group sizes (realistic)

# Generate scores with realistic means and standard deviations per department
# np.random.normal(mean, std, size) draws samples from a normal distribution
score_params = {
    "Computer Science": (74, 12),   # mean=74, std=12 — wide spread
    "Mathematics":      (81, 8),    # mean=81, std=8  — tighter, higher scores
    "Physics":          (70, 15),   # mean=70, std=15 — widest variation
    "Biology":          (77, 10),   # mean=77, std=10 — moderate spread
}

# Build the records list by generating scores for each department
records = []
for dept, n in zip(departments, n_per_dept):
    mean, std = score_params[dept]
    scores = np.random.normal(mean, std, n)          # generate n scores
    scores = np.clip(scores, 0, 100).round(1)        # clip to [0,100] range, round
    for score in scores:
        records.append({"Department": dept, "Score": score})

# Convert list of dictionaries to a DataFrame — standard pandas pattern
df_scores = pd.DataFrame(records)

# Also add a Pass/Fail label for a third variable we will use later
# Students scoring ≥ 60 pass; otherwise they fail
df_scores["Result"] = df_scores["Score"].apply(lambda s: "Pass" if s >= 60 else "Fail")

print("Dataset Preview (first 10 rows):")
print(df_scores.head(10).to_string(index=False))
print(f"\nTotal students: {len(df_scores)}")
print("\nDepartment summary:")
print(df_scores.groupby("Department")["Score"].describe().round(2))


In [ ]:
# ─── Basic Swarm Plot ────────────────────────────────────────────────────────
# sns.swarmplot() places each data point as a dot, nudging overlapping points
# sideways so every single observation remains visible.

plt.figure(figsize=(11, 6))

# The core call:
#   data=   → our DataFrame
#   x=      → the categorical grouping variable (department on X-axis)
#   y=      → the numeric variable to display (exam score on Y-axis)
#   size=   → dot diameter in points (larger = easier to see, but more overlap)
#   palette=→ color scheme name from seaborn (one color per category)
#   edgecolor / linewidth → thin dark border makes overlapping points distinguishable
sns.swarmplot(
    data=df_scores,
    x="Department",
    y="Score",
    size=5,                    # dot size; 4–6 is usually ideal for moderate datasets
    palette="Set2",            # harmonious qualitative color palette (8 distinct colors)
    edgecolor="gray",          # dot border color to separate touching points
    linewidth=0.4              # border thickness in points
)

# Reference line at the passing threshold
plt.axhline(y=60, color="red", linestyle="--", linewidth=1.5, label="Pass threshold (60)")

# Labels and title
plt.title("Student Exam Scores by Department — Basic Swarm Plot",
          fontsize=15, pad=16)
plt.xlabel("Department", fontsize=12)
plt.ylabel("Exam Score (out of 100)", fontsize=12)
plt.legend(fontsize=10)
plt.ylim(0, 110)             # extra headroom above 100 for readability
plt.tight_layout()
plt.show()

print("What we can read instantly:")
print("  • Mathematics scores cluster HIGH (most dots near top)")
print("  • Physics has the WIDEST spread — dots fan out the most")
print("  • Individual outliers are visible as isolated dots")
print("  • Nearly all students passed (very few dots below the red line)")


In [ ]:
# ─── Swarm + Box Overlay — Best of Both Worlds ──────────────────────────────
# Overlaying a box plot on a swarm plot gives us BOTH statistical summaries
# (median, IQR, whiskers) AND individual point visibility.
# This is a professional best practice for small-to-medium datasets.

plt.figure(figsize=(11, 7))

# ── Step 1: Draw the box plot FIRST (it sits behind the swarm) ────────────────
# width=0.4  → narrower box so swarm dots have room beside it
# fliersize=0 → hide the box plot's own outlier dots; the swarm shows all points
# boxprops, medianprops, whiskerprops → fine-grained style control via dictionaries
sns.boxplot(
    data=df_scores,
    x="Department",
    y="Score",
    width=0.42,
    fliersize=0,                              # suppress default outlier markers
    palette="pastel",                         # light pastel colors for the box (won't hide dots)
    boxprops=dict(alpha=0.55),               # semi-transparent box fill
    medianprops=dict(color="black", linewidth=2.5),   # bold median line
    whiskerprops=dict(linestyle="--", linewidth=1.2), # dashed whiskers
    capprops=dict(linewidth=1.5)             # cap line at whisker ends
)

# ── Step 2: Draw the swarm plot ON TOP ───────────────────────────────────────
# Using dodge=False because we are not splitting by a hue variable here
sns.swarmplot(
    data=df_scores,
    x="Department",
    y="Score",
    size=5,
    palette="Set2",        # slightly different palette than the box — adds contrast
    edgecolor="gray",
    linewidth=0.4,
    zorder=2               # zorder=2 ensures swarm dots render above box elements
)

plt.axhline(y=60, color="red", linestyle="--", linewidth=1.5,
            alpha=0.8, label="Pass threshold (60)")

plt.title("Exam Score Distribution — Swarm + Box Overlay",
          fontsize=16, pad=18)
plt.xlabel("Department", fontsize=12)
plt.ylabel("Exam Score", fontsize=12)
plt.ylim(0, 112)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

print("Reading this combined chart:")
print("  • Box shows: median line, 25th–75th percentile range (the box body), whiskers")
print("  • Swarm shows: exact position of every student's score")
print("  • Together: we see both summary statistics AND the full data landscape")


In [ ]:
# ─── Color Encoding a Third Variable (Pass / Fail) ──────────────────────────
# Adding a hue= argument colors each dot by the Result column (Pass/Fail).
# This adds a THIRD dimension to the chart without adding a new axis.
# dodge=True separates the two groups slightly to reduce overlap.

plt.figure(figsize=(12, 7))

# hue=     → column used to assign colors
# hue_order → explicit order of hue categories (controls which color maps to which)
# dodge=   → if True, points in different hue groups are offset from each other
# palette  → dictionary mapping category names to specific colors
sns.swarmplot(
    data=df_scores,
    x="Department",
    y="Score",
    hue="Result",
    hue_order=["Pass", "Fail"],         # Pass listed first → gets first palette color
    dodge=True,                          # separate pass and fail dots side by side
    palette={"Pass": "seagreen", "Fail": "tomato"},  # meaningful colors (green=good, red=bad)
    size=5,
    edgecolor="gray",
    linewidth=0.4
)

# Threshold line
plt.axhline(y=60, color="black", linestyle=":", linewidth=1.5, alpha=0.7,
            label="Pass threshold (60)")

plt.title("Exam Scores Colored by Pass / Fail Outcome",
          fontsize=16, pad=18)
plt.xlabel("Department", fontsize=12)
plt.ylabel("Exam Score", fontsize=12)
plt.ylim(0, 112)

# Move legend outside the plot area so it doesn't block data
plt.legend(title="Result", title_fontsize=11, fontsize=10,
           bbox_to_anchor=(1.01, 1), loc="upper left", borderaxespad=0)
plt.tight_layout()
plt.show()

print("Insights with hue encoding:")
print("  • Red dots (failures) stand out immediately — no calculation needed")
print("  • Physics has the most red dots — widest spread includes lower scores")
print("  • Mathematics has almost no failures — tight high cluster")
print("  • Biology and CS show a few scattered failures near the threshold")


In [ ]:
# ─── Swarm vs Box vs Violin — Visual Comparison ─────────────────────────────
# When should you pick each? Seeing them side-by-side on the SAME data is
# the clearest way to understand the trade-offs.

fig, axes = plt.subplots(1, 3, figsize=(18, 7))

# ── Panel 1: Box Plot ─────────────────────────────────────────────────────────
sns.boxplot(
    data=df_scores, x="Department", y="Score",
    palette="Set2", ax=axes[0], width=0.5, fliersize=4
)
axes[0].axhline(60, color="red", linestyle="--", linewidth=1.2)
axes[0].set_title("Box Plot
Shows statistics; hides individuals",
                  fontsize=13, pad=10)
axes[0].set_xlabel("Department")
axes[0].set_ylabel("Score")
axes[0].set_ylim(0, 112)
# Rotate long department names 15 degrees to prevent overlap
axes[0].tick_params(axis="x", rotation=15)

# ── Panel 2: Violin Plot ──────────────────────────────────────────────────────
sns.violinplot(
    data=df_scores, x="Department", y="Score",
    palette="Set2", ax=axes[1], inner="box",   # inner="box" adds mini box inside violin
    cut=0                                        # cut=0 limits violin to actual data range
)
axes[1].axhline(60, color="red", linestyle="--", linewidth=1.2)
axes[1].set_title("Violin Plot
Shows distribution shape; no individuals",
                  fontsize=13, pad=10)
axes[1].set_xlabel("Department")
axes[1].set_ylabel("")           # suppress repeated y-axis label
axes[1].set_ylim(0, 112)
axes[1].tick_params(axis="x", rotation=15)

# ── Panel 3: Swarm Plot ───────────────────────────────────────────────────────
sns.swarmplot(
    data=df_scores, x="Department", y="Score",
    palette="Set2", ax=axes[2], size=5,
    edgecolor="gray", linewidth=0.4
)
axes[2].axhline(60, color="red", linestyle="--", linewidth=1.2)
axes[2].set_title("Swarm Plot
Shows EVERY individual point",
                  fontsize=13, pad=10)
axes[2].set_xlabel("Department")
axes[2].set_ylabel("")
axes[2].set_ylim(0, 112)
axes[2].tick_params(axis="x", rotation=15)

fig.suptitle("Same Data — Three Different Perspectives", fontsize=17, y=1.01)
plt.tight_layout()
plt.show()

print("Choose your plot type based on your priority:")
print("  Box   → Quick statistical summaries; large datasets (n > 1000)")
print("  Violin → Distribution shape; moderate data; no need to see individuals")
print("  Swarm → Every point matters; small-medium data; stakeholders need full story")


In [ ]:
# ─── Crowding: When Swarm Plots Struggle ─────────────────────────────────────
# As sample size grows, dots pile up and become indistinguishable.
# This cell demonstrates the crowding problem and a practical solution.

# Generate a LARGE dataset (500 students) to stress-test the swarm plot
np.random.seed(7)
large_scores = []
for dept, n in zip(departments, [140, 120, 130, 110]):
    mean, std = score_params[dept]
    sc = np.clip(np.random.normal(mean, std, n), 0, 100).round(1)
    for s in sc:
        large_scores.append({"Department": dept, "Score": s})
df_large = pd.DataFrame(large_scores)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Left: Crowded swarm (large dataset) ──────────────────────────────────────
sns.swarmplot(
    data=df_large, x="Department", y="Score",
    size=3,            # smaller dots help, but crowding still occurs
    palette="Set2", ax=axes[0], edgecolor="none"
)
axes[0].set_title(f"Swarm with n={len(df_large)} — Crowding Problem",
                  fontsize=13, color="crimson")
axes[0].set_xlabel("Department")
axes[0].set_ylabel("Score")
axes[0].tick_params(axis="x", rotation=15)

# ── Right: Strip + violin as the recommended solution ────────────────────────
# Strip plot is like swarm but randomly jitters points horizontally (faster for big data)
sns.violinplot(
    data=df_large, x="Department", y="Score",
    palette="pastel", ax=axes[1], inner=None,    # inner=None removes internal markers
    cut=0, alpha=0.6
)
sns.stripplot(
    data=df_large, x="Department", y="Score",
    color="black", alpha=0.25, size=2.5,          # semi-transparent small dots
    jitter=True,                                   # jitter=True adds random horizontal offset
    ax=axes[1]
)
axes[1].set_title(f"Violin + Strip with n={len(df_large)} — Better for Large Data",
                  fontsize=13, color="green")
axes[1].set_xlabel("Department")
axes[1].set_ylabel("")
axes[1].tick_params(axis="x", rotation=15)

plt.suptitle("Crowding: When to Switch Away from Swarm Plot", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print("Practical guideline:")
print("  n < 150  → Swarm plot works well")
print("  n 150-500 → Swarm with small dots (size=2-3); consider strip+violin")
print("  n > 500  → Violin + transparent strip; or box + jitter; skip pure swarm")


---
# Guided Lab — Build It Yourself 

Now it's your turn! Work through the tasks below using the provided datasets and starter templates.

## Task 1 — Horizontal Bar Chart 

**Scenario:** You are a data analyst at a streaming platform. You have weekly watch-time data (hours) for different genres.

### Steps
1. Create the `df_genres` DataFrame from the data provided in the starter cell
2. Sort by watch-time **descending** (remember the `ascending=True` trick for `barh()`)
3. Plot a horizontal bar chart with:
   - A meaningful title and axis labels
   - Value labels on each bar
   - A vertical line showing the average watch-time
4. **Discuss:** Which 3 genres should the platform promote to maximise engagement?

---

## Task 2 — Swarm Plot 

**Scenario:** A gym collected fitness test scores (0–100) for members across 3 training programs.

### Steps
1. Use the `df_gym` DataFrame from the starter cell
2. Create a **swarm + box overlay** chart (like Cell 12 in the demo)
3. Color the dots by `"Level"` (Beginner / Intermediate / Advanced)
4. **Discuss:** Which program shows the most consistent improvement? How can you tell from the swarm?

In [ ]:
# ─── YOUR CODE: Task 1 — Streaming Genre Horizontal Bar Chart ───────────────

# ── Starter Data ─────────────────────────────────────────────────────────────
genre_data = {
    "Drama":         1240,
    "Action":        1085,
    "Comedy":         980,
    "Documentary":    760,
    "Thriller":       870,
    "Romance":        540,
    "Sci-Fi":         620,
    "Animation":      430,
    "Horror":         380,
    "Reality TV":     290,
}
df_genres = pd.DataFrame(
    list(genre_data.items()),
    columns=["Genre", "Watch_Hours"]
)
print("Genres dataset:")
print(df_genres.to_string(index=False))

# ── YOUR PLOTTING CODE BELOW ──────────────────────────────────────────────────

# Step 1: Sort the DataFrame for correct barh() ordering
# HINT: sort ascending=True so the highest value appears at the TOP of the chart
df_genres_sorted = df_genres.sort_values("Watch_Hours", ascending=___)  # fill in True or False

# Step 2: Create the figure
plt.figure(figsize=(11, 7))

# Step 3: Draw horizontal bars
# HINT: plt.barh(y_labels, x_values, color=..., edgecolor=..., height=...)
bars = plt.barh(
    df_genres_sorted["Genre"],
    df_genres_sorted["Watch_Hours"],
    color=___,          # choose a color, e.g. "coral" or "#4A90D9"
    edgecolor="white",
    height=0.65
)

# Step 4: Add value labels
for bar in bars:
    width = bar.get_width()
    # HINT: plt.text(x, y, label_string, va="center", ha="left", ...)
    plt.text(___, ___, f"{width:,} hrs", va="center", ha="left", fontsize=10)

# Step 5: Add average reference line
avg_hours = df_genres["Watch_Hours"].mean()
plt.axvline(x=avg_hours, color="red", linestyle="--", linewidth=1.5,
            label=f"Average: {avg_hours:.0f} hrs")

# Step 6: Labels, title, legend
plt.title("___ YOUR TITLE HERE ___", fontsize=16)
plt.xlabel("Weekly Watch Hours")
plt.ylabel("Genre")
plt.legend()
plt.grid(axis="x", alpha=0.3)
plt.xlim(0, 1450)
plt.tight_layout()
plt.show()


In [ ]:
# ─── YOUR CODE: Task 2 — Gym Fitness Score Swarm + Box Chart ────────────────

# ── Starter Data ─────────────────────────────────────────────────────────────
np.random.seed(99)   # fixed seed so everyone gets the same dataset

programs = ["Cardio Focus", "Strength Training", "Mixed HIIT"]
levels    = ["Beginner", "Intermediate", "Advanced"]

gym_records = []
program_params = {
    "Cardio Focus":      {"Beginner": (55, 10), "Intermediate": (68, 8),  "Advanced": (80, 6)},
    "Strength Training": {"Beginner": (50, 12), "Intermediate": (65, 9),  "Advanced": (82, 7)},
    "Mixed HIIT":        {"Beginner": (60, 9),  "Intermediate": (74, 7),  "Advanced": (88, 5)},
}
for prog in programs:
    for level in levels:
        mean, std = program_params[prog][level]
        n = 18   # 18 members per program-level combination
        scores = np.clip(np.random.normal(mean, std, n), 0, 100).round(1)
        for s in scores:
            gym_records.append({"Program": prog, "Level": level, "Score": s})

df_gym = pd.DataFrame(gym_records)
print("Gym dataset sample:")
print(df_gym.head(9).to_string(index=False))
print(f"Total members: {len(df_gym)}")

# ── YOUR PLOTTING CODE BELOW ──────────────────────────────────────────────────

plt.figure(figsize=(12, 7))

# Step 1: Draw box plot first (it sits behind the swarm)
# HINT: sns.boxplot(data=..., x="Program", y="Score", width=0.4, fliersize=0, ...)
sns.boxplot(
    data=df_gym,
    x=___,
    y="Score",
    width=0.42,
    fliersize=0,
    palette="pastel",
    boxprops=dict(alpha=0.5),
    medianprops=dict(color="black", linewidth=2.5)
)

# Step 2: Draw swarm plot on top with hue="Level"
# HINT: sns.swarmplot(data=..., x="Program", y="Score", hue="Level",
#                     dodge=True, palette=..., size=5, ...)
sns.swarmplot(
    data=df_gym,
    x="Program",
    y="Score",
    hue=___,
    hue_order=levels,
    dodge=___,    # True or False? What does it do?
    palette="Set1",
    size=5,
    edgecolor="gray",
    linewidth=0.4,
    zorder=2
)

# Step 3: Labels, title, legend
plt.title("___ YOUR TITLE HERE ___", fontsize=16)
plt.xlabel("Training Program")
plt.ylabel("Fitness Score")
plt.legend(title="Level", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.ylim(0, 110)
plt.tight_layout()
plt.show()

# Discussion prompt
print("Discuss with your partner:")
print("  1. Which program has the widest gap between Beginner and Advanced?")
print("  2. Which program produces the most CONSISTENT scores across levels?")
print("  3. Why is the swarm more informative than just a box plot here?")


---
# Challenge / Extension Task (10 minutes)

**Scenario:** You work for a tech company comparing software engineers' salaries
across 5 cities and 3 experience levels (Junior / Mid / Senior).

## Your Tasks
1. Generate the dataset using `np.random.normal()` with realistic salary means
   - Junior: city means in range \$55k–\$80k, std \$8k
   - Mid: city means in range \$85k–\$115k, std \$10k
   - Senior: city means in range \$120k–\$160k, std \$15k
2. Create a **horizontal bar chart** showing the **Top 5 highest-paying city × level combinations**
3. Create a **swarm plot** showing salary distributions by city, colored by experience level
4. Answer in a markdown cell:
   - Which city has the highest salary ceiling for Senior engineers?
   - Which experience level shows the most salary variation across cities?
   - In your swarm plot, what does clustering at the top tell you about a city?

*Bonus:* Combine both charts into a single `plt.subplots(1, 2)` figure.

In [ ]:
# ─── Challenge Starter Code ──────────────────────────────────────────────────
# Use this as your launching pad — add or modify as needed.

np.random.seed(2024)   # keep results reproducible

cities = ["New York", "San Francisco", "Austin", "Chicago", "Seattle"]
levels_sal = ["Junior", "Mid", "Senior"]

# Define realistic salary means per city per level (in thousands $)
salary_means = {
    "New York":      {"Junior": 78, "Mid": 112, "Senior": 158},
    "San Francisco": {"Junior": 82, "Mid": 118, "Senior": 165},
    "Austin":        {"Junior": 65, "Mid":  92, "Senior": 132},
    "Chicago":       {"Junior": 68, "Mid":  98, "Senior": 138},
    "Seattle":       {"Junior": 75, "Mid": 108, "Senior": 152},
}
salary_stds = {"Junior": 8, "Mid": 10, "Senior": 15}
n_each = 20   # 20 engineers per city-level cell

salary_records = []
for city in cities:
    for level in levels_sal:
        mean = salary_means[city][level]
        std  = salary_stds[level]
        salaries = np.clip(np.random.normal(mean, std, n_each), 45, 200).round(1)
        for sal in salaries:
            salary_records.append({"City": city, "Level": level, "Salary_k": sal})

df_salary = pd.DataFrame(salary_records)
print("Salary dataset preview:")
print(df_salary.sample(8).to_string(index=False))
print(f"\nTotal records: {len(df_salary)}")

# ── WRITE YOUR VISUALIZATION CODE BELOW ──────────────────────────────────────
# Chart 1: Horizontal bar of Top-5 city-level combinations by median salary
# Chart 2: Swarm plot of salary by city, hue=Level

# Tip for Chart 1:
# df_salary.groupby(["City","Level"])["Salary_k"].median().reset_index()
# will give you a single median per city-level combination


---
# Summary & Key Takeaways

Let us lock in what we covered today:

## Horizontal Bar Charts
- Use **`barh()`** whenever category labels are long or numerous
- **Always sort** — ascending sort + `barh()` places the highest value at the top
- Use **Top-N filtering** to focus on what matters; don't show everything if it adds noise
- **Label bars directly** — remove the reader's need to look at the axis scale
- **Single purposeful color or gradient** — never rainbow colormaps for ranked data

## Swarm Plots
- Use **`sns.swarmplot()`** when your dataset is small-to-medium (n < ~150) and every point matters
- **Overlay with box plot** for the power combo: statistics + individual visibility
- Use **`hue=`** to encode a third categorical variable with color
- Use **`dodge=True`** to separate hue groups and reduce overlap
- **Beware crowding** — for n > 500, switch to violin + strip plot

## When to Use Which
| Chart | Best For |
|-------|----------|
| Horizontal Bar | Rankings, comparisons, long labels, top-N summaries |
| Swarm Plot | Small datasets, individual point visibility, distribution + outlier context |

## Best Practices Recap
1. Always label axes, add a title, and include a legend when using color encoding
2. Add reference lines (mean, threshold) to give readers an anchor
3. Choose color palettes intentionally — qualitative for categories, diverging for correlations
4. Consider your audience: presentations need fewer details than exploratory analysis

---

---


Answer these 3 questions (write answers in the code cell below as `print()` statements or in a new markdown cell):

1. **Horizontal Bar:** Name ONE scenario where you would use a horizontal bar chart instead of a vertical one, and explain why.
2. **Swarm Plot:** What is the main advantage of a swarm plot over a box plot for a dataset with n = 50?
3. **Personal Reflection:** What was the most surprising thing you learned today?

*Your instructor will collect these responses as a quick formative assessment.*

In [ ]:
# Write your answers here ──────────────────────────────────
# Replace each placeholder string with your actual answer.

answer_1 = """
YOUR ANSWER:
I would use a horizontal bar chart when...
"""

answer_2 = """
YOUR ANSWER:
The main advantage of a swarm plot over a box plot for n=50 is...
"""

answer_3 = """
YOUR ANSWER:
The most surprising thing I learned today was...
"""

# Print formatted responses
print("=" * 60)
print("EXIT TICKET RESPONSES")
print("=" * 60)
print(f"Q1 — When to use Horizontal Bar:\n{answer_1.strip()}")
print("-" * 60)
print(f"Q2 — Swarm vs Box advantage:\n{answer_2.strip()}")
print("-" * 60)
print(f"Q3 — Personal reflection:\n{answer_3.strip()}")
print("=" * 60)

In [ ]:
# ─── Quick-Reference Helper Functions ────────────────────────────────────────
# Reusable utility functions you can copy into any future notebook.

def ranked_barh(df, label_col, value_col, top_n=None, color="steelblue", title=""):
    """
    Create a sorted horizontal bar chart with value labels.

    Parameters
    ----------
    df        : pandas DataFrame containing the data
    label_col : str — column name for bar labels (Y-axis)
    value_col : str — column name for bar values (X-axis)
    top_n     : int or None — if set, show only top N rows
    color     : str — bar fill color
    title     : str — chart title
    """
    data = df.sort_values(value_col, ascending=False)
    if top_n:
        data = data.head(top_n)
    data = data.sort_values(value_col, ascending=True)  # flip for barh top-rank-at-top

    plt.figure(figsize=(11, max(5, len(data) * 0.45)))
    bars = plt.barh(data[label_col], data[value_col], color=color,
                    edgecolor="white", height=0.65)
    for bar in bars:
        w = bar.get_width()
        plt.text(w * 1.01, bar.get_y() + bar.get_height() / 2,
                 f"{w:,.1f}", va="center", ha="left", fontsize=9)
    plt.title(title, fontsize=14, pad=14)
    plt.xlabel(value_col)
    plt.ylabel(label_col)
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()


def swarm_box(df, x_col, y_col, hue_col=None, palette="Set2", title=""):
    """
    Create a swarm plot overlaid on a box plot.

    Parameters
    ----------
    df      : pandas DataFrame
    x_col   : str — categorical column for X-axis grouping
    y_col   : str — numeric column for Y-axis values
    hue_col : str or None — optional column for color encoding
    palette : str — seaborn color palette name
    title   : str — chart title
    """
    plt.figure(figsize=(12, 7))
    sns.boxplot(data=df, x=x_col, y=y_col, width=0.42, fliersize=0,
                palette="pastel",
                boxprops=dict(alpha=0.55),
                medianprops=dict(color="black", linewidth=2.5))
    sns.swarmplot(data=df, x=x_col, y=y_col, hue=hue_col,
                  dodge=(hue_col is not None),
                  palette=palette, size=5,
                  edgecolor="gray", linewidth=0.4, zorder=2)
    plt.title(title, fontsize=15, pad=16)
    plt.xlabel(x_col)
    plt.ylabel(y_col)
    if hue_col:
        plt.legend(title=hue_col, bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.tight_layout()
    plt.show()


print("Helper functions loaded!")
print("  ranked_barh(df, label_col, value_col, top_n, color, title)")
print("  swarm_box  (df, x_col, y_col, hue_col, palette, title)")
